In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
# Load and process the CSV file
labels_df_israeli = pd.read_csv(r"C:\Users\waleed\Desktop\Ai\lesson11\day14\traffic sign.csv")

# Create the class labels dictionary from the CSV data
class_labels_israeli = dict(zip(labels_df_israeli['ClassID'], labels_df_israeli['Name']))

# Verify the data is loaded correctly
print("Number of classes:", len(class_labels_israeli))
print("Sample of labels:", dict(list(class_labels_israeli.items())[:3]))

# Create the final class labels dictionary
class_labels = {**class_labels_israeli}

In [ ]:
import os
file_path = r"C:\Users\waleed\Desktop\Ai\lesson11\day14\traffic sign.csv"
print(f"File exists: {os.path.exists(file_path)}")

In [ ]:

# Then create the class_labels dictionary
class_labels = {**class_labels_israeli}

In [ ]:
# Define the paths to the datasets
ISRAELI_DATASET_PATH = r"C:\Users\waleed\Desktop\final_project\IsraeliData"  

In [ ]:
# Custom dataset class
class TrafficSignDataset(Dataset):
    def __init__(self, dataset_path, transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        for class_id in os.listdir(dataset_path):
            class_folder_path = os.path.join(dataset_path, class_id)
            if os.path.isdir(class_folder_path):
                for img_file in os.listdir(class_folder_path):
                    img_path = os.path.join(class_folder_path, img_file)
                    try:
                        image = Image.open(img_path).convert('RGB')
                        if self.transform:
                            image = self.transform(image)
                        self.images.append(image)
                        self.labels.append(int(class_id))
                    except Exception as e:
                        print(f"Error loading image {img_path}: {e}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# Image transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])


train_dataset_israeli = TrafficSignDataset(ISRAELI_DATASET_PATH, transform=transform)


train_dataset = train_dataset_israeli

In [ ]:
# Split the dataset into training, validation, and test sets
train_size = int(0.7 * len(train_dataset))
val_size = int(0.2 * len(train_dataset))
test_size = len(train_dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size, test_size])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [13]:
# Define the CNN model using nn.Sequential
class TrafficSignModel(nn.Module):
    def __init__(self, num_classes):
        super(TrafficSignModel, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 6 * 6, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# Instantiate the model
num_classes = len(class_labels)
model = TrafficSignModel(num_classes)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

NameError: name 'class_labels' is not defined

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=40):
    train_losses, val_losses, val_accuracies = [], [], []
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to('cpu'), labels.to('cpu')
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
        
        # Validation step
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to('cpu'), labels.to('cpu')
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        # Calculate metrics
        train_loss /= len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)
        val_accuracy = 100 * correct / total
        
        # Append metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
        
        # Print epoch results
        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%')
    
    return train_losses, val_losses, val_accuracies


In [ ]:
# Train the model
#train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=40)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_results(train_losses, val_losses, val_accuracies):
    epochs = range(1, len(train_losses) + 1)
    
    # Plot loss
    plt.figure(figsize=(12, 6))
    plt.plot(epochs, train_losses, label="Training Loss")
    plt.plot(epochs, val_losses, label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Loss over Epochs")
    plt.legend()
    plt.show()
    
    # Plot accuracy
    plt.figure(figsize=(12, 6))
    plt.plot(epochs, val_accuracies, label="Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.title("Validation Accuracy over Epochs")
    plt.legend()
    plt.show()

In [ ]:
train_losses, val_losses, val_accuracies = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=40)
plot_training_results(train_losses, val_losses, val_accuracies)

In [ ]:
# Function to evaluate the model on the test set
def evaluate_model(model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to('cpu'), labels.to('cpu')
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100 * correct / total
    print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.2f}%')

# Evaluate the model on the test set
evaluate_model(model, test_loader, criterion)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def plot_confusion_matrix(model, data_loader, num_classes, device="cpu", figsize=(12, 12)):
    """
    Plot a larger confusion matrix with class numbers instead of class names.
    
    Args:
        model: Trained PyTorch model.
        data_loader: DataLoader for the dataset to evaluate.
        num_classes: Number of classes in the dataset.
        device: Device to run the evaluation on ("cpu" or "cuda").
        figsize: Tuple specifying the figure size (width, height).
    """
    model.eval()
    all_labels, all_predictions = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
    
    # Generate confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Plot the confusion matrix
    fig, ax = plt.subplots(figsize=figsize)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(num_classes)))
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45, ax=ax)
    plt.title("Confusion Matrix", fontsize=16)
    plt.xlabel("Predicted Label (Class Number)", fontsize=14)
    plt.ylabel("True Label (Class Number)", fontsize=14)
    plt.show()


In [ ]:
plot_confusion_matrix(model, test_loader, num_classes=len(class_labels),figsize= (16,16))

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def plot_roc_curve(model, data_loader, num_classes, device="cpu"):
    model.eval()
    all_labels, all_outputs = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            all_labels.extend(labels.cpu().numpy())
            all_outputs.extend(outputs.cpu().numpy())
    
    # Binarize the labels for multi-class ROC
    all_labels_bin = label_binarize(all_labels, classes=range(num_classes))
    fpr, tpr, roc_auc = {}, {}, {}
    
    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(all_labels_bin[:, i], np.array(all_outputs)[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # Plot the ROC curve
    plt.figure(figsize=(12, 6))
    for i in range(num_classes):
        plt.plot(fpr[i], tpr[i], label=f"Class {i} (AUC = {roc_auc[i]:.2f})")
    
    plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend(loc="best")
    plt.show()

In [ ]:
plot_roc_curve(model, test_loader, num_classes=len(class_labels))

In [ ]:
# Function to test the model on a new image
import torch
import os
from PIL import Image
def predict_traffic_sign(image_path):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image = transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    model.to(device)
    with torch.no_grad():
        output = model(image)
        predicted_class = torch.argmax(output, dim=1).item()
        if predicted_class not in class_labels:
            raise ValueError(f"Predicted class {predicted_class} not found in class_labels.")
        return class_labels[predicted_class]

In [ ]:
# Example prediction
print(predict_traffic_sign(r"C:\Users\waleed\Desktop\final_project\IsraeliData\31\Screenshot 2024-09-25 191645.jpg"))
